# EfficientNet-B1 (HED color space) — Ordinal Focal Loss

This notebook trains a single **EfficientNet-B1** on the PANDAS ISUP-grading task using the
**HED (Haematoxylin–Eosin–DAB) stain color space** instead of RGB, then evaluates it on the
validation and test sets.

The hyperparameters are **not re-searched** — we reuse the best Optuna configuration already
found for B1 in `efficientnet-family-optuna-ordinal-focal.ipynb` (see `logs/family-b1-results.txt`).
The only change vs. that pipeline is the color space: every patch is converted RGB → HED right
before the mosaic is assembled.

Pipeline per backbone:
1. **Noise cleaning** — drop the noisiest 20% of images (highest `difficulty_score` in `entropy.csv`).
2. **HED transform** — convert each patch to the HED stain space (applied *after* RGB-space photometric augmentations).
3. **Full training** with the reused best params (cosine schedule + warm-up, AMP, early stopping on QWK).
4. **Evaluation** on the validation set and on the held-out `data/test.csv` with bootstrap 95% CIs.

> Artifacts are written with the `b1-hed` prefix (`logs/b1-hed-*`, `models/b1-hed.pth`) so they do
> **not** overwrite the RGB family results.


## Imports

In [ ]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from torch.amp import autocast, GradScaler
import torchvision.models as tvm
import albumentations as Albu
from albumentations.core.transforms_interface import ImageOnlyTransform
from skimage import color
from skimage.exposure import rescale_intensity
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display
import sys
sys.path.append('../..')
from utils.dataset import PandasOverlapDataset


## Fixed Configuration

In [ ]:
SEED           = 42
NUM_WORKERS    = 4
OUTPUT_CLASSES = 5       # ordinal thresholds for ISUP 0-5
WARMUP_EPOCHS  = 1
WARMUP_FACTOR  = 2
USE_AMP        = True    # mixed precision (fp16) — major VRAM saver
VAL_FOLD       = 3       # held-out fold used for evaluation
ENTROPY_DROP_FRAC = 0.20 # fraction of noisiest images removed

# --- Training budget (matches the family notebook's full-training settings) ---
N_EPOCHS_FULL  = 40      # epochs for the final training
PATIENCE       = 8       # early-stopping patience

# --- Reused best Optuna hyperparameters for EfficientNet-B1 ---------------------
# Copied verbatim from logs/family-b1-results.txt (study best val QWK = 0.8665).
BEST_PARAMS_B1 = {
    'lr':              0.00015930522616241006,
    'dropout_rate':    0.4540362888980227,
    'focal_gamma':     1.0617534828874073,
    'focal_alpha':     0.8759278817295955,
    'unfreeze_blocks': 5,
    'weight_decay':    4.335281794951567e-06,
    'batch_size':      8,
}
STUDY_BEST_B1 = 0.8665

NAME = 'b1'        # backbone key in the registry / build_model
TAG  = 'b1-hed'    # filename prefix for all artifacts

REPO_DIR   = '../..'                                   # pos-propose/family -> repo root
DATA_DIR   = os.path.join(REPO_DIR, 'data')
IMAGES_DIR = os.path.join(REPO_DIR, '..', 'bag_of_patches')

LOG_DIR    = 'logs'
MODEL_DIR  = 'models'
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


## VRAM Utility Helpers

In [ ]:
def free_vram(*objs):
    """Aggressive cleanup: delete refs, run gc, empty CUDA cache."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def vram_report(tag=''):
    if not torch.cuda.is_available():
        return
    alloc    = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    peak     = torch.cuda.max_memory_allocated() / 1e9
    print(f'  [VRAM {tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB')


## Loss Function — Ordinal Focal Loss

In [ ]:
class OrdinalFocalLoss(nn.Module):
    """Focal BCE over ordinal thresholds + a soft penalty on the expected-class distance."""

    def __init__(self, alpha=0.25, gamma=2.0, ordinal_weight=0.2, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ordinal_weight = ordinal_weight
        self.reduction = reduction

    def forward(self, logits, targets):
        # numerical stability under AMP
        logits  = logits.float()
        targets = targets.float()

        probs = torch.sigmoid(logits)
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')

        # focal term
        p_t  = probs * targets + (1 - probs) * (1 - targets)
        loss = self.alpha * ((1 - p_t) ** self.gamma) * bce

        if self.reduction == 'mean':
            focal_loss = loss.mean()
        elif self.reduction == 'sum':
            focal_loss = loss.sum()
        else:
            focal_loss = loss

        # ordinal penalty on the (soft) predicted class index
        expected_class = probs.sum(dim=1)
        target_class   = targets.sum(dim=1)
        max_class      = logits.shape[1]
        ordinal_loss   = ((expected_class - target_class) ** 2).mean() / (max_class ** 2)

        return focal_loss + self.ordinal_weight * ordinal_loss


def decode_ordinal_predictions(logits):
    """sigmoid -> threshold @ 0.5 -> sum  ==>  ISUP grade 0..5."""
    return (torch.sigmoid(logits.float()) > 0.5).sum(dim=1)


## HED Color-Space Transform

`skimage.color.rgb2hed` returns float stain densities (Haematoxylin / Eosin / DAB). We rescale each
channel per-image to `uint8 [0, 255]` so the HED patch flows through the **exact same** mosaic /
white-padding / tensor pipeline used for RGB in the family notebook — no other code path changes.

The transform is placed **last** in the augmentation `Compose`, so the RGB-space photometric
augmentations (brightness/contrast, hue/saturation) still operate on RGB before the conversion.
Geometric augmentations (flips/transpose) commute with the color conversion.

In [ ]:
class RGB2HEDTransform(ImageOnlyTransform):
    """RGB patch -> HED stain space, per-channel rescaled to uint8 [0, 255]."""

    def __init__(self, p=1.0):
        super().__init__(p=p)

    def apply(self, image, **params):
        if image.ndim == 3 and image.shape[2] == 4:   # drop alpha if present
            image = image[:, :, :3]
        hed = color.rgb2hed(image)                     # HWC float64 stain densities
        out = np.empty(hed.shape, dtype=np.uint8)
        for c in range(3):
            ch = rescale_intensity(hed[:, :, c], out_range=(0.0, 1.0))
            out[:, :, c] = (ch * 255.0).astype(np.uint8)
        return out


## EfficientNet Wrapper + Backbone Registry

In [ ]:
EFFICIENTNET_REGISTRY = {
    # 'b0': (tvm.efficientnet_b0, tvm.EfficientNet_B0_Weights.DEFAULT),
    'b1': (tvm.efficientnet_b1, tvm.EfficientNet_B1_Weights.DEFAULT),
    # 'b2': (tvm.efficientnet_b2, tvm.EfficientNet_B2_Weights.DEFAULT),
}


class EfficientNetApi(nn.Module):
    """
    Generic torchvision EfficientNet (b0-b7) wrapper.

    Freezes the whole backbone, then unfreezes the last `unfreeze_blocks` feature
    blocks. The classifier is replaced by an ordinal-regression head.
    """
    def __init__(self, model, output_dimensions, dropout_rate=0.4, unfreeze_blocks=2):
        super().__init__()
        self.model = model

        for p in self.model.parameters():
            p.requires_grad = False

        if unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True

        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features
        self.model.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, output_dimensions),
        )

    def extract(self, x):
        x = self.model(x)
        if x.ndim == 4:                 # safety for any non-pooled output
            x = x.mean(dim=[2, 3])
        return x

    def forward(self, x):
        return self.head(self.extract(x))


def build_model(name, dropout_rate, unfreeze_blocks):
    ctor, weights = EFFICIENTNET_REGISTRY[name]
    backbone = ctor(weights=weights)
    return EfficientNetApi(backbone, OUTPUT_CLASSES, dropout_rate, unfreeze_blocks).to(device)


## Data Loading with Entropy-based Noise Cleaning

In [ ]:
df_all = pd.read_csv(os.path.join(DATA_DIR, 'train_5fold.csv'))
df_all.columns = df_all.columns.str.strip()
print(f'Total records: {len(df_all)}')

# --- Noise cleaning: drop the noisiest images by difficulty score ---
df_entropy = pd.read_csv(os.path.join(DATA_DIR, 'entropy.csv'))
df_entropy = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove   = int(len(df_entropy) * ENTROPY_DROP_FRAC)
noisy_ids  = set(df_entropy.head(n_remove)['image_id'])
df_all     = df_all[~df_all['image_id'].isin(noisy_ids)].reset_index(drop=True)
print(f'After entropy filter (removed {n_remove} noisiest): {len(df_all)}')


def drop_missing(df):
    exists = df['image_id'].apply(lambda x: os.path.isdir(os.path.join(IMAGES_DIR, str(x))))
    return df[exists].reset_index(drop=True)


df_train = drop_missing(df_all[df_all['fold'] != VAL_FOLD].reset_index(drop=True))
df_val   = drop_missing(df_all[df_all['fold'] == VAL_FOLD].reset_index(drop=True))

print(f'Train: {len(df_train)}   Val: {len(df_val)}')
print('Val class distribution:')
print(df_val['isup_grade'].value_counts().sort_index())


## Augmentation

The RGB-space photometric/geometric augmentations are kept identical to the family notebook; the
`RGB2HEDTransform` is appended **last** so it converts the already-augmented RGB patch to HED. The
validation/test pipeline applies only the HED conversion (no augmentation).

In [ ]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomBrightnessContrast(p=0.3),
    Albu.HueSaturationValue(p=0.2),
    RGB2HEDTransform(p=1.0),
])

val_transforms = Albu.Compose([
    RGB2HEDTransform(p=1.0),
])


## Data Loaders + Train/Val Epoch Helpers (AMP-enabled)

In [ ]:
def make_loaders(batch_size):
    train_ds = PandasOverlapDataset(IMAGES_DIR, df_train, transforms=train_transforms, overlap=0)
    val_ds   = PandasOverlapDataset(IMAGES_DIR, df_val,   transforms=val_transforms,   overlap=0)

    common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=RandomSampler(train_ds), **common)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **common)
    return train_loader, val_loader


def run_epoch_train(model, loader, optimizer, loss_fn, device, scaler, accum_steps=1):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, targets, _) in enumerate(tqdm(loader, desc='Train', leave=False)):
        imgs    = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(device_type='cuda', enabled=USE_AMP, dtype=torch.float16):
            logits = model(imgs)
            loss   = loss_fn(logits, targets) / accum_steps

        scaler.scale(loss).backward()
        if (step + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * accum_steps)

    return float(np.mean(losses))


def run_epoch_val(model, loader, loss_fn, device):
    model.eval()
    losses, preds, gts = [], [], []
    with torch.no_grad():
        for imgs, targets, _ in tqdm(loader, desc='Val', leave=False):
            imgs    = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=USE_AMP, dtype=torch.float16):
                logits = model(imgs)
                loss   = loss_fn(logits, targets)
            losses.append(loss.item())
            preds.append(decode_ordinal_predictions(logits).cpu())
            gts.append(targets.sum(1).long().cpu())

    preds = torch.cat(preds).numpy()
    gts   = torch.cat(gts).numpy()
    return dict(
        val_loss=float(np.mean(losses)),
        val_kappa=cohen_kappa_score(gts, preds, weights='quadratic'),
        val_acc=accuracy_score(gts, preds),
        val_f1=f1_score(gts, preds, average='macro', zero_division=0),
    )


## Full Training (reused best params) + Validation Evaluation

In [ ]:
def run_full_training(name, best_params, log_path, model_path):
    free_vram()
    model   = build_model(name, best_params['dropout_rate'], best_params['unfreeze_blocks'])
    loss_fn = OrdinalFocalLoss(alpha=best_params['focal_alpha'], gamma=best_params['focal_gamma'])
    scaler  = GradScaler(enabled=USE_AMP)
    train_loader, val_loader = make_loaders(best_params['batch_size'])

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=best_params['lr'] / WARMUP_FACTOR, weight_decay=best_params['weight_decay'],
    )
    scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_FULL - WARMUP_EPOCHS)
    scheduler     = GradualWarmupScheduler(optimizer, multiplier=WARMUP_FACTOR,
                                           total_epoch=WARMUP_EPOCHS, after_scheduler=scheduler_cos)

    history = dict(train_loss=[], val_loss=[], val_kappa=[], val_acc=[], val_f1=[])
    best_kappa, best_epoch, no_improve = -1.0, 0, 0
    open(log_path, 'w').close()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, N_EPOCHS_FULL + 1):
        train_loss = run_epoch_train(model, train_loader, optimizer, loss_fn, device, scaler)
        metrics    = run_epoch_val(model, val_loader, loss_fn, device)
        scheduler.step()
        lr_now = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        for k in ('val_loss', 'val_kappa', 'val_acc', 'val_f1'):
            history[k].append(metrics[k])

        with open(log_path, 'a') as f:
            f.write(
                f'epoch: {epoch} | lr: {lr_now:.7f} | train_loss: {train_loss:.5f} | '
                f'val_loss: {metrics["val_loss"]:.5f} | val_kappa: {metrics["val_kappa"]:.4f} | '
                f'val_acc: {metrics["val_acc"]:.4f}\n'
            )
        print(f'  [{name}] epoch {epoch:02d}/{N_EPOCHS_FULL}  train={train_loss:.4f}  '
              f'val_loss={metrics["val_loss"]:.4f}  QWK={metrics["val_kappa"]:.4f}  '
              f'acc={metrics["val_acc"]*100:.2f}%')

        if metrics['val_kappa'] > best_kappa:
            best_kappa, best_epoch, no_improve = metrics['val_kappa'], epoch, 0
            torch.save(model.state_dict(), model_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  [{name}] early stop @ epoch {epoch} (best QWK={best_kappa:.4f} @ epoch {best_epoch})')
                break

    free_vram(model, optimizer)
    return history, best_kappa, best_epoch


def evaluate_on_val(name, model_path, best_params, n_boot=1000):
    free_vram()
    model = build_model(name, best_params['dropout_rate'], best_params['unfreeze_blocks'])
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
    _, val_loader = make_loaders(best_params['batch_size'])

    preds, gts = [], []
    with torch.no_grad():
        for imgs, targets, _ in tqdm(val_loader, desc=f'Eval {name}', leave=False):
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=USE_AMP, dtype=torch.float16):
                logits = model(imgs)
            preds.append(decode_ordinal_predictions(logits).cpu())
            gts.append(targets.sum(1).long())
    preds = torch.cat(preds).numpy()
    gts   = torch.cat(gts).numpy()

    rng = np.random.default_rng(SEED)
    n   = len(gts)
    bacc = np.empty(n_boot); bkap = np.empty(n_boot); bf1 = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        bacc[i] = accuracy_score(gts[idx], preds[idx])
        bkap[i] = cohen_kappa_score(gts[idx], preds[idx], weights='quadratic')
        bf1[i]  = f1_score(gts[idx], preds[idx], average='macro', zero_division=0)

    res = dict(
        acc=accuracy_score(gts, preds),
        kappa=cohen_kappa_score(gts, preds, weights='quadratic'),
        f1=f1_score(gts, preds, average='macro', zero_division=0),
        acc_ci=(np.percentile(bacc, 2.5), np.percentile(bacc, 97.5)),
        kappa_ci=(np.percentile(bkap, 2.5), np.percentile(bkap, 97.5)),
        f1_ci=(np.percentile(bf1, 2.5), np.percentile(bf1, 97.5)),
        acc_std=bacc.std(ddof=1), kappa_std=bkap.std(ddof=1), f1_std=bf1.std(ddof=1),
        preds=preds, gts=gts,
    )
    free_vram(model)
    return res


## Per-model Artifacts (curves, confusion matrix, results txt)

In [ ]:
LABELS = [f'ISUP {i}' for i in range(6)]


def save_model_artifacts(name, tag, history, best_epoch, res, best_params, study_best):
    # ── training curves ──────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    axes[0, 0].plot(history['train_loss'], label='Train')
    axes[0, 0].plot(history['val_loss'],   label='Val')
    axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True)

    axes[0, 1].plot(history['val_kappa'], color='orange')
    axes[0, 1].axvline(best_epoch - 1, color='red', ls='--', label=f'best ep {best_epoch}')
    axes[0, 1].set_title('Val QWK'); axes[0, 1].legend(); axes[0, 1].grid(True)

    axes[1, 0].plot(history['val_acc'], color='green')
    axes[1, 0].set_title('Val Accuracy'); axes[1, 0].grid(True)

    axes[1, 1].plot(history['val_f1'], color='red')
    axes[1, 1].set_title('Val Macro F1'); axes[1, 1].grid(True)

    plt.suptitle(f'EfficientNet-{name.upper()} (HED) — Ordinal Focal Loss', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, f'{tag}-training.png'), dpi=200, bbox_inches='tight')
    plt.show()

    # ── confusion matrix ─────────────────────────────────────────────
    cm      = confusion_matrix(res['gts'], res['preds'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
    axes[0].set_title('Confusion (counts)'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
    axes[1].set_title('Confusion (normalized)'); axes[1].set_ylabel('True'); axes[1].set_xlabel('Pred')
    plt.suptitle(f'EfficientNet-{name.upper()} (HED) — Validation', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, f'{tag}-confusion-matrix.png'), dpi=200, bbox_inches='tight')
    plt.show()

    # ── results txt ──────────────────────────────────────────────────
    with open(os.path.join(LOG_DIR, f'{tag}-results.txt'), 'w') as f:
        f.write(f'EfficientNet-{name.upper()} (HED color space) + Ordinal Focal Loss\n')
        f.write('=' * 70 + '\n\nReused best Optuna hyperparameters (from RGB B1 search):\n')
        for k, v in best_params.items():
            f.write(f'  {k}: {v}\n')
        f.write(f'  (RGB study best val QWK during HPO: {study_best:.4f})\n\n')
        f.write('Validation-set results (bootstrap 1000, 95% CI):\n')
        f.write(f'  Accuracy : {res["acc"]*100:.2f}% ± {res["acc_std"]*100:.2f}%  '
                f'[{res["acc_ci"][0]*100:.2f}%-{res["acc_ci"][1]*100:.2f}%]\n')
        f.write(f'  QW Kappa : {res["kappa"]:.4f} ± {res["kappa_std"]:.4f}  '
                f'[{res["kappa_ci"][0]:.4f}-{res["kappa_ci"][1]:.4f}]\n')
        f.write(f'  Macro F1 : {res["f1"]:.4f} ± {res["f1_std"]:.4f}  '
                f'[{res["f1_ci"][0]:.4f}-{res["f1_ci"][1]:.4f}]\n\n')
        f.write('Classification Report:\n')
        f.write(classification_report(res['gts'], res['preds'],
                                      target_names=LABELS, digits=4, zero_division=0))
        f.write('\nConfusion Matrix:\n' + str(cm) + '\n')


## Train EfficientNet-B1 (HED) + Validation Evaluation

Trains with the reused best B1 params on the HED-converted patches, then evaluates the best
checkpoint on the validation set and writes all artifacts under the `b1-hed` prefix.

In [ ]:

log_path   = os.path.join(LOG_DIR, f'{TAG}.txt')
model_path = os.path.join(MODEL_DIR, f'{TAG}.pth')

print('=' * 72)
print(f'  Training EfficientNet-{NAME.upper()} on HED color space')
print('  Best params:', BEST_PARAMS_B1)
print('=' * 72)

history, best_kappa, best_epoch = run_full_training(NAME, BEST_PARAMS_B1, log_path, model_path)
print(f'\nBest val QWK during training: {best_kappa:.4f} @ epoch {best_epoch}')

res = evaluate_on_val(NAME, model_path, BEST_PARAMS_B1, n_boot=1000)
print(f'\nValidation (best checkpoint):')
print(f'  Accuracy : {res["acc"]*100:.2f}%  [{res["acc_ci"][0]*100:.2f}%-{res["acc_ci"][1]*100:.2f}%]')
print(f'  QW Kappa : {res["kappa"]:.4f}  [{res["kappa_ci"][0]:.4f}-{res["kappa_ci"][1]:.4f}]')
print(f'  Macro F1 : {res["f1"]:.4f}  [{res["f1_ci"][0]:.4f}-{res["f1_ci"][1]:.4f}]')

save_model_artifacts(NAME, TAG, history, best_epoch, res, BEST_PARAMS_B1, STUDY_BEST_B1)
print('\nArtifacts saved with prefix:', TAG)


## Test Set Evaluation

Evaluates the trained HED B1 checkpoint on the independent `data/test.csv` (HED transform applied,
no augmentation) with bootstrap 95% CIs, a confusion matrix and a classification report.

In [ ]:
print('=' * 72)
print(f'  TEST SET EVALUATION — EfficientNet-{NAME.upper()} (HED)')
print('=' * 72)

df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
df_test.columns = df_test.columns.str.strip()
df_test = drop_missing(df_test)
print(f'Test records: {len(df_test)}')

test_ds     = PandasOverlapDataset(IMAGES_DIR, df_test, transforms=val_transforms, overlap=0)
test_loader = DataLoader(test_ds, batch_size=BEST_PARAMS_B1['batch_size'], shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

free_vram()
model = build_model(NAME, BEST_PARAMS_B1['dropout_rate'], BEST_PARAMS_B1['unfreeze_blocks'])
model.load_state_dict(torch.load(model_path, weights_only=True))
model.eval()

preds, gts = [], []
with torch.no_grad():
    for imgs, targets, _ in tqdm(test_loader, desc=f'Test {NAME}', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=USE_AMP, dtype=torch.float16):
            logits = model(imgs)
        preds.append(decode_ordinal_predictions(logits).cpu())
        gts.append(targets.sum(1).long().cpu())

preds = torch.cat(preds).numpy()
gts   = torch.cat(gts).numpy()
free_vram(model)

# bootstrap CIs
rng = np.random.default_rng(SEED)
n   = len(gts)
n_boot = 1000
bacc = np.empty(n_boot); bkap = np.empty(n_boot); bf1 = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.integers(0, n, n)
    bacc[i] = accuracy_score(gts[idx], preds[idx])
    bkap[i] = cohen_kappa_score(gts[idx], preds[idx], weights='quadratic')
    bf1[i]  = f1_score(gts[idx], preds[idx], average='macro', zero_division=0)

test_acc   = accuracy_score(gts, preds)
test_kappa = cohen_kappa_score(gts, preds, weights='quadratic')
test_f1    = f1_score(gts, preds, average='macro', zero_division=0)

print(f'\n>>> TEST EfficientNet-{NAME.upper()} (HED)')
print(f'  Accuracy : {test_acc*100:.2f}%  [{np.percentile(bacc,2.5)*100:.2f}%-{np.percentile(bacc,97.5)*100:.2f}%]')
print(f'  QW Kappa : {test_kappa:.4f}  [{np.percentile(bkap,2.5):.4f}-{np.percentile(bkap,97.5):.4f}]')
print(f'  Macro F1 : {test_f1:.4f}  [{np.percentile(bf1,2.5):.4f}-{np.percentile(bf1,97.5):.4f}]')

# confusion matrix
cm      = confusion_matrix(gts, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title('Test Confusion (counts)'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
axes[1].set_title('Test Confusion (normalized)'); axes[1].set_ylabel('True'); axes[1].set_xlabel('Pred')
plt.suptitle(f'EfficientNet-{NAME.upper()} (HED) — Test Set', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'{TAG}-test-confusion-matrix.png'), dpi=200, bbox_inches='tight')
plt.show()

print('\nClassification Report (test):')
print(classification_report(gts, preds, target_names=LABELS, digits=4, zero_division=0))

with open(os.path.join(LOG_DIR, f'{TAG}-test-results.txt'), 'w') as f:
    f.write(f'EfficientNet-{NAME.upper()} (HED color space) — Test Set Results\n')
    f.write('=' * 70 + '\n\n')
    f.write('Test-set results (bootstrap 1000, 95% CI):\n')
    f.write(f'  Accuracy : {test_acc*100:.2f}% ± {bacc.std(ddof=1)*100:.2f}%  '
            f'[{np.percentile(bacc,2.5)*100:.2f}%-{np.percentile(bacc,97.5)*100:.2f}%]\n')
    f.write(f'  QW Kappa : {test_kappa:.4f} ± {bkap.std(ddof=1):.4f}  '
            f'[{np.percentile(bkap,2.5):.4f}-{np.percentile(bkap,97.5):.4f}]\n')
    f.write(f'  Macro F1 : {test_f1:.4f} ± {bf1.std(ddof=1):.4f}  '
            f'[{np.percentile(bf1,2.5):.4f}-{np.percentile(bf1,97.5):.4f}]\n\n')
    f.write('Classification Report:\n')
    f.write(classification_report(gts, preds, target_names=LABELS, digits=4, zero_division=0))
    f.write('\nConfusion Matrix:\n' + str(cm) + '\n')

print('\nTest artifacts saved with prefix:', TAG)
